# Climate Forecasts

1. DATASETS: [Hugging Face: Howard Tsai | All](https://huggingface.co/Howard881010)
2. DATASETS: [Hugging Face: Howard Tsai | Climate-1 Day](https://huggingface.co/datasets/Howard881010/climate-1day)
3. REPO: [Multimodal_Forecasting](https://github.com/Rose-STL-Lab/Multimodal_Forecasting/tree/main)
    - Get data from here.


In [1]:
import os, sys

import pandas as pd

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:
pd.set_option('max_colwidth', 800)

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
dataset_path = os.path.join(base_data_path, 'forecast_bench/2024-07-21-human.json')

In [4]:
df = DataProcessing.load_from_file(dataset_path, file_type='json')
df

,forecast_due_date,question_set,questions
0,2024-07-21,2024-07-21-human.json,"{'id': 'TPkEjiNb1wVCIGFnPcDD', 'source': 'manifold', 'question': 'Will the average global temperature in 2024 exceed 2023?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/SteveRabin/will-the-average-global-temperature.', 'background': '2023 is trending to be the hottest year on record.', 'market_info_open_datetime': '2023-10-13T17:44:30+00:00', 'market_info_close_datetime': '2025-01-01T04:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.markets/SteveRabin/will-the-average-global-temperature', 'freeze_datetime': '2024-07-12T00:00:00+00:00', 'freeze_datetime_value': '0.7565624485542961', 'freeze_datetime_value_explanation': 'The market value.', 'source_intro': 'We would like you to predict the outcome of ..."
1,2024-07-21,2024-07-21-human.json,"{'id': 'KCFbp1TH0RYN4j5zYdmh', 'source': 'manifold', 'question': 'Will any event of the Paris 2024 Olympic Games be postponed or cancelled due to the water quality of the Seine?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/Pob/will-any-event-of-the-paris-2024-ol.', 'background': 'It also counts as YES if an event is moved to a venue other than the Seine river ahead of time because of water quality concerns. Background: Seine River events scrapped again, renewing doubts over Paris Olympics plan Paris Olympics promises Seine clean-up after pollution spoils triathlon | Financial Times', 'market_info_open_datetime': '2023-08-25T19:59:35+00:00', 'market_info_close_datetime': '2024-08-11T21:59:00+00:00', 'market_info_resolution_criter..."
2,2024-07-21,2024-07-21-human.json,"{'id': 'q6wJThcy6TJnQKrbjALm', 'source': 'manifold', 'question': 'Will the majority of new cars sold be electric vehicles by the end of 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/TobiasSowaaed/will-the-majority-of-new-cars-sold-c4d566afc8c7.', 'background': 'Majority is 50%+ and it's worldwide marketshare', 'market_info_open_datetime': '2023-09-01T23:59:55+00:00', 'market_info_close_datetime': '2030-12-31T22:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.markets/TobiasSowaaed/will-the-majority-of-new-cars-sold-c4d566afc8c7', 'freeze_datetime': '2024-07-12T00:00:00+00:00', 'freeze_datetime_value': '0.6912524093033491', 'freeze_datetime_value_explanation': 'The market value.', 'source_intro':..."
3,2024-07-21,2024-07-21-human.json,"{'id': 'cLxfTH5GDfmqeNJwClVF', 'source': 'manifold', 'question': 'Will the majority of new cars sold worldwide be electric before the end of 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/kian_spire/will-the-majority-of-new-cars-sold.', 'background': 'This question will resolve as YES if more than 50% of new cars sold worldwide in the year 2030 are electric. The source for determining the percentage of electric cars sold will be the International Energy Agency (IEA) annual report on electric vehicles. The report will be released in 2031 and will provide data on the percentage of electric cars sold in 2030. This question does not include hybrid cars, only fully electric cars. The question does not consider any potential chang..."
4,2024-07-21,2024-07-21-human.json,"{'id': 'ygVTjZRGDZvp0EQJ1L53', 'source': 'manifold', 'question': 'Will sports betting be legal in California and Texas by 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/10thOfficial/will-sports-betting-be-legal-in-cal.', 'background': 'Must be able to place both an in-person and online sports bet legally in each state on January 1, 2030.', 'market_info_open_datetime': '2024-01-05T22:00:15+00:00', 'market_info_close_datetime': '2030-01-01T04:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://mani

In [39]:
result = []

for idx, row in df.iterrows():
    forecast_due = row["forecast_due_date"]
    # Skip the whole row if the forecast date is “N/A”
    if forecast_due == "N/A":
        continue

    # Build a base meta dict that will be copied for each variant
    base_meta = {
        "meta_data": row["questions"],
        "forecast_due_date": forecast_due,
        "question_set": row["question_set"],
    }

    question = row["questions"]["question"]

    # Determine if we need to duplicate the row per resolution date
    if "{resolution_date}" in question:
        for rd in row["questions"]["resolution_dates"]:
            q = question.replace("{forecast_due_date}", forecast_due).replace(
                "{resolution_date}", rd
            )
            result.append(
                {**base_meta, "question": q}
            )
    else:
        # No resolution placeholder – just replace the forecast placeholder
        question = question.replace("{forecast_due_date}", forecast_due)
        result.append(
            {**base_meta, "question": question}
        )

# The list `result` now holds all the expanded question / meta‑data pairs
questions_and_meta_data = result

In [40]:
pd.DataFrame(questions_and_meta_data)

,meta_data,forecast_due_date,question_set,question
0,"{'id': 'TPkEjiNb1wVCIGFnPcDD', 'source': 'manifold', 'question': 'Will the average global temperature in 2024 exceed 2023?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/SteveRabin/will-the-average-global-temperature.', 'background': '2023 is trending to be the hottest year on record.', 'market_info_open_datetime': '2023-10-13T17:44:30+00:00', 'market_info_close_datetime': '2025-01-01T04:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.markets/SteveRabin/will-the-average-global-temperature', 'freeze_datetime': '2024-07-12T00:00:00+00:00', 'freeze_datetime_value': '0.7565624485542961', 'freeze_datetime_value_explanation': 'The market value.', 'source_intro': 'We would like you to predict the outcome of ...",2024-07-21,2024-07-21-human.json,Will the average global temperature in 2024 exceed 2023?
1,"{'id': 'KCFbp1TH0RYN4j5zYdmh', 'source': 'manifold', 'question': 'Will any event of the Paris 2024 Olympic Games be postponed or cancelled due to the water quality of the Seine?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/Pob/will-any-event-of-the-paris-2024-ol.', 'background': 'It also counts as YES if an event is moved to a venue other than the Seine river ahead of time because of water quality concerns. Background: Seine River events scrapped again, renewing doubts over Paris Olympics plan Paris Olympics promises Seine clean-up after pollution spoils triathlon | Financial Times', 'market_info_open_datetime': '2023-08-25T19:59:35+00:00', 'market_info_close_datetime': '2024-08-11T21:59:00+00:00', 'market_info_resolution_criter...",2024-07-21,2024-07-21-human.json,Will any event of the Paris 2024 Olympic Games be postponed or cancelled due to the water quality of the Seine?
2,"{'id': 'q6wJThcy6TJnQKrbjALm', 'source': 'manifold', 'question': 'Will the majority of new cars sold be electric vehicles by the end of 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/TobiasSowaaed/will-the-majority-of-new-cars-sold-c4d566afc8c7.', 'background': 'Majority is 50%+ and it's worldwide marketshare', 'market_info_open_datetime': '2023-09-01T23:59:55+00:00', 'market_info_close_datetime': '2030-12-31T22:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.markets/TobiasSowaaed/will-the-majority-of-new-cars-sold-c4d566afc8c7', 'freeze_datetime': '2024-07-12T00:00:00+00:00', 'freeze_datetime_value': '0.6912524093033491', 'freeze_datetime_value_explanation': 'The market value.', 'source_intro':...",2024-07-21,2024-07-21-human.json,Will the majority of new cars sold be electric vehicles by the end of 2030?
3,"{'id': 'cLxfTH5GDfmqeNJwClVF', 'source': 'manifold', 'question': 'Will the majority of new cars sold worldwide be electric before the end of 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/kian_spire/will-the-majority-of-new-cars-sold.', 'background': 'This question will resolve as YES if more than 50% of new cars sold worldwide in the year 2030 are electric. The source for determining the percentage of electric cars sold will be the International Energy Agency (IEA) annual report on electric vehicles. The report will be released in 2031 and will provide data on the percentage of electric cars sold in 2030. This question does not include hybrid cars, only fully electric cars. The question does not consider any potential chang...",2024-07-21,2024-07-21-human.json,Will the majority of new cars sold worldwide be electric before the end of 2030?
4,"{'id': 'ygVTjZRGDZvp0EQJ1L53', 'source': 'manifold', 'question': 'Will sports betting be legal in California and Texas by 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/10thOfficial/will-sports-betting-be-legal-in-cal.'

Will HUM's market close price on {resolution_date} be higher than its market close price on {forecast_due_date}?\n\nStock splits and reverse splits will be accounted for in resolving this question. Forecasts on questions about companies that have been delisted (through mergers or bankruptcy) will resolve to their final close price.	

In [37]:
questions_and_meta_data = []

for idx, row in df.iterrows():
    forecast_due_date = row['forecast_due_date']
    question_set = row['question_set']
    question = row['questions']['question']
    resolution_dates = row['questions']['resolution_dates']
    print(resolution_dates)

    print(question)


    if 'forecast_due_date' in question or 'resolution_date' in question:

        resolution_date_questions = []
        for resolution_date in resolution_dates:
            question = question.replace("{resolution_date}", resolution_date)
            question_and_meta_data = {
            'question': question
        }

    else:
        question_and_meta_data = {
            'question': question
        }

        question_and_meta_data = {
            'meta_data': row['questions'],
            'forecast_due_date': forecast_due_date,
            'question_set': question_set
        }
    questions_and_meta_data.append(question_and_meta_data)

questions_and_meta_data

N/A
Will the average global temperature in 2024 exceed 2023?
N/A
Will any event of the Paris 2024 Olympic Games be postponed or cancelled due to the water quality of the Seine?
N/A
Will the majority of new cars sold be electric vehicles by the end of 2030?
N/A
Will the majority of new cars sold worldwide be electric before the end of 2030?
N/A
Will sports betting be legal in California and Texas by 2030?
N/A
👍👎 Manifold Leadership Approval Rating [Market Index]
N/A
Will Rafael Nadal win another Grand Slam before his retirement?
N/A
[2024 Formula 1 Season] Will Oscar Piastri or Daniel Ricciardo win a race?
N/A
Will Ding Liren repeat as World Chess Champion?
N/A
Will Lionel Messi retire before the end of 2024?
N/A
Will Rational Animations' video about Manifold Markets reach 200k views by the end of 2024?
N/A
Will over 70% of Manifold users think that Elon Musk does not deserve 100 billion dollars?
N/A
Will the U.S. win the gold medal in the Men's Basketball at the 2024 Summer Olympics in

[{'meta_data': {'id': 'TPkEjiNb1wVCIGFnPcDD',
   'source': 'manifold',
   'question': 'Will the average global temperature in 2024 exceed 2023?',
   'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/SteveRabin/will-the-average-global-temperature.',
   'background': '2023 is trending to be the hottest year on record.',
   'market_info_open_datetime': '2023-10-13T17:44:30+00:00',
   'market_info_close_datetime': '2025-01-01T04:59:00+00:00',
   'market_info_resolution_criteria': 'N/A',
   'url': 'https://manifold.markets/SteveRabin/will-the-average-global-temperature',
   'freeze_datetime': '2024-07-12T00:00:00+00:00',
   'freeze_datetime_value': '0.7565624485542961',
   'freeze_datetime_value_explanation': 'The market value.',
   'source_intro': "We would like you to predict the outcome of a prediction market. A prediction market, in this context, is the aggregate of predictions submitted by users on the website Manifold. You're going to pr

In [38]:
pd.DataFrame(questions_and_meta_data)

,meta_data,forecast_due_date,question_set,question
0,"{'id': 'TPkEjiNb1wVCIGFnPcDD', 'source': 'manifold', 'question': 'Will the average global temperature in 2024 exceed 2023?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/SteveRabin/will-the-average-global-temperature.', 'background': '2023 is trending to be the hottest year on record.', 'market_info_open_datetime': '2023-10-13T17:44:30+00:00', 'market_info_close_datetime': '2025-01-01T04:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.markets/SteveRabin/will-the-average-global-temperature', 'freeze_datetime': '2024-07-12T00:00:00+00:00', 'freeze_datetime_value': '0.7565624485542961', 'freeze_datetime_value_explanation': 'The market value.', 'source_intro': 'We would like you to predict the outcome of ...",2024-07-21,2024-07-21-human.json,NaN
1,"{'id': 'KCFbp1TH0RYN4j5zYdmh', 'source': 'manifold', 'question': 'Will any event of the Paris 2024 Olympic Games be postponed or cancelled due to the water quality of the Seine?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/Pob/will-any-event-of-the-paris-2024-ol.', 'background': 'It also counts as YES if an event is moved to a venue other than the Seine river ahead of time because of water quality concerns. Background: Seine River events scrapped again, renewing doubts over Paris Olympics plan Paris Olympics promises Seine clean-up after pollution spoils triathlon | Financial Times', 'market_info_open_datetime': '2023-08-25T19:59:35+00:00', 'market_info_close_datetime': '2024-08-11T21:59:00+00:00', 'market_info_resolution_criter...",2024-07-21,2024-07-21-human.json,NaN
2,"{'id': 'q6wJThcy6TJnQKrbjALm', 'source': 'manifold', 'question': 'Will the majority of new cars sold be electric vehicles by the end of 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/TobiasSowaaed/will-the-majority-of-new-cars-sold-c4d566afc8c7.', 'background': 'Majority is 50%+ and it's worldwide marketshare', 'market_info_open_datetime': '2023-09-01T23:59:55+00:00', 'market_info_close_datetime': '2030-12-31T22:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.markets/TobiasSowaaed/will-the-majority-of-new-cars-sold-c4d566afc8c7', 'freeze_datetime': '2024-07-12T00:00:00+00:00', 'freeze_datetime_value': '0.6912524093033491', 'freeze_datetime_value_explanation': 'The market value.', 'source_intro':...",2024-07-21,2024-07-21-human.json,NaN
3,"{'id': 'cLxfTH5GDfmqeNJwClVF', 'source': 'manifold', 'question': 'Will the majority of new cars sold worldwide be electric before the end of 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/kian_spire/will-the-majority-of-new-cars-sold.', 'background': 'This question will resolve as YES if more than 50% of new cars sold worldwide in the year 2030 are electric. The source for determining the percentage of electric cars sold will be the International Energy Agency (IEA) annual report on electric vehicles. The report will be released in 2031 and will provide data on the percentage of electric cars sold in 2030. This question does not include hybrid cars, only fully electric cars. The question does not consider any potential chang...",2024-07-21,2024-07-21-human.json,NaN
4,"{'id': 'ygVTjZRGDZvp0EQJ1L53', 'source': 'manifold', 'question': 'Will sports betting be legal in California and Texas by 2030?', 'resolution_criteria': 'Resolves to the outcome of the question found at https://manifold.markets/10thOfficial/will-sports-betting-be-legal-in-cal.', 'background': 'Must be able to place both an in-person and online sports bet legally in each state on January 1, 2030.', 'market_info_open_datetime': '2024-01-05T22:00:15+00:00', 'market_info_close_datetime': '2030-01-01T04:59:00+00:00', 'market_info_resolution_criteria': 'N/A', 'url': 'https://manifold.mar